# Análise da evolução populacional no Brasil

Comparação dos dados populacionais de 2010 e 2022, com organização dos resultados por unidade federativa e município.

In [ ]:
import pandas as pd

arquivo = 'CD2022_Populacao_2010_Compatibilizada_20231222.xlsx'

# Leitura e preparação dos dados
dados = pd.read_excel(arquivo, skiprows=2)
dados = dados.dropna(subset=['COD. UF']).copy()

dados = dados.rename(columns={
    dados.columns[0]: 'DESCARTAR',
    'UF': 'ESTADO',
    'COD. UF': 'CODIGO_UF',
    'COD. MUNIC': 'CODIGO_MUNICIPIO',
    'NOME DO MUNICÍPIO': 'MUNICIPIO'
})

# Padroniza as colunas de população
dados.columns = ['DESCARTAR', 'ESTADO', 'CODIGO_UF', 'CODIGO_MUNICIPIO',
                 'MUNICIPIO', 'POPULACAO_2010_SINOPSE',
                 'POPULACAO_2010', 'POPULACAO_2022']

dados['DIFERENCA_POPULACIONAL'] = dados['POPULACAO_2022'] - dados['POPULACAO_2010']
dados['VARIACAO_PERCENTUAL'] = (
    dados['DIFERENCA_POPULACIONAL'] / dados['POPULACAO_2010'] * 100
).round(2)

dados.head()


In [ ]:
# Resumo da população por estado
resumo_estados = (
    dados.groupby('ESTADO', as_index=False)
    .agg(
        POPULACAO_2010=('POPULACAO_2010', 'sum'),
        POPULACAO_2022=('POPULACAO_2022', 'sum'),
        DIFERENCA_POPULACIONAL=('DIFERENCA_POPULACIONAL', 'sum')
    )
)

resumo_estados['VARIACAO_PERCENTUAL'] = (
    resumo_estados['DIFERENCA_POPULACIONAL'] / resumo_estados['POPULACAO_2010'] * 100
).round(2)

resumo_estados = resumo_estados.sort_values(
    by='DIFERENCA_POPULACIONAL', ascending=False
)

resumo_estados.to_csv(
    'resumo_populacao_estados.csv', index=False, sep=';', encoding='utf-8-sig'
)

resumo_estados


In [ ]:
# Ranking dos municípios conforme a diferença de população
colunas_municipios = [
    'ESTADO', 'CODIGO_UF', 'CODIGO_MUNICIPIO', 'MUNICIPIO',
    'POPULACAO_2010', 'POPULACAO_2022',
    'DIFERENCA_POPULACIONAL', 'VARIACAO_PERCENTUAL'
]

ranking_municipios = dados[colunas_municipios].sort_values(
    by='DIFERENCA_POPULACIONAL', ascending=False
).reset_index(drop=True)

ranking_municipios.to_csv(
    'ranking_populacao_municipios.csv', index=False, sep=';', encoding='utf-8-sig'
)

ranking_municipios.head(10)
